In [6]:
class OuterJob:
    def __init__(self, n_jobs, n_buffers):
        self.n_jobs = n_jobs
        self.n_buffers = n_buffers
        self.phase = [0] * n_buffers   # one phase per buffer

    def load_into(self, job_idx, buffer_idx):
        old = self.phase[buffer_idx]
        new = old ^ 1
        self.phase[buffer_idx] = new
        return f"O{job_idx}:B{buffer_idx} p{old}→{new}"

    
    def wait_parity(self, buffer_idx, parity):
        current = self.phase[buffer_idx]
        return f"W B{buffer_idx} wait_p{parity} (cur_p{current})"
      
      
class InnerJob:
    def __init__(self, n_jobs, n_buffers):
        self.n_jobs = n_jobs
        self.n_buffers = n_buffers

    def load_into(self, job_idx, outer_buffer_idx, buffer_idx):
        return f"I{job_idx}:b{buffer_idx}, OB{outer_buffer_idx}"

    def compute(self, job_idx, buffer_idx):
        return f"b{buffer_idx}→C{job_idx}"
      
   
def print_step(step, outer=None, inner_load=None, inner_compute=None):
    left  = f"{outer:<18}" if outer else " " * 18
    mid   = f"{inner_load:<10}" if inner_load else " " * 10
    right = f"{inner_compute:<10}" if inner_compute else " " * 10
    print(f"{step:04d} | {left} | {mid} | {right}")


N_outer_jobs = 8
N_outer_stages = 3
N_inner_jobs = 7
N_inner_stages = 4
jobs_completed = 0
jobs_loaded = 0

O_JOB = OuterJob(N_outer_jobs, N_outer_stages)
I_JOB = InnerJob(N_inner_jobs, N_inner_stages)





In [12]:
step = 0
for i in range(N_outer_stages): 
  outer, inner_load, inner_comp = None,None,None 
  outer = O_JOB.load_into(i,i)
  print_step(step, outer, inner_load, inner_comp)
  step += 1
  
print(O_JOB.wait_parity(0,0))

for i in range(N_inner_stages-1): 
  outer, inner_load, inner_comp = None,None,None 
  inner_load = I_JOB.load_into(i,0,i) 
  print_step(step, outer, inner_load, inner_comp)
  step += 1
  

num_outer_full = N_outer_jobs - N_outer_stages 


print("-----------------------------------------------------------------")
for outer_idx in range(num_outer_full): 
  #curr consume stage = outer_idx % N_outer_stages 
  #next consume and wait stage = (outer_idx + 1) % N_outer_stages 
  #if next consume and wit stage == 0 flip parity my love. 
  
  out_cons_stage = outer_idx % N_outer_stages 
  next_out_cons_stage = (outer_idx + 1) % N_outer_stages
  next_out_load_idx = (outer_idx + (N_outer_stages))
  next_out_load_stage = next_out_load_idx % N_outer_stages   
  
  parity = (next_out_load_idx//N_outer_stages) % 2

  for (inner_idx) in range(N_inner_jobs - (N_inner_stages-1)): 
    outer, inner_load, inner_comp = None,None,None 
    inner_load_idx = inner_idx + (N_inner_stages-1)
    inner_load_stage = ((outer_idx*N_inner_jobs) + inner_load_idx) % N_inner_stages
    inner_compute_stage = ((outer_idx*N_inner_jobs) + inner_idx) % N_inner_stages
    inner_load = I_JOB.load_into(inner_load_idx,out_cons_stage, inner_load_stage)
    inner_comp = I_JOB.compute(inner_idx,inner_compute_stage)
    print_step(step, outer, inner_load, inner_comp)
    step += 1
  
  outer, inner_load, inner_comp = None,None,None 
  outer = O_JOB.load_into(next_out_load_idx, next_out_load_stage)
  #print_step(step, outer, inner_load, inner_comp)
  #step += 1
  
  for (inner_idx) in range(N_inner_jobs - (N_inner_stages-1), N_inner_jobs): 
    
    inner_load_idx = inner_idx + (N_inner_stages-1)
    inner_load_stage = ((outer_idx*N_inner_jobs) + inner_load_idx) % N_inner_stages
    inner_compute_stage = ((outer_idx*N_inner_jobs) + inner_idx) % N_inner_stages
    inner_load = I_JOB.load_into(inner_load_idx,out_cons_stage, inner_load_stage)
    inner_comp = I_JOB.compute(inner_idx,inner_compute_stage)
    print_step(step, outer, inner_load, inner_comp)
    step += 1
    outer, inner_load, inner_comp = None,None,None 
    
  print("-----------------------------------------------------------------")
  
  
    
  

0000 | O0:B0 p1→0         |            |           
0001 | O1:B1 p1→0         |            |           
0002 | O2:B2 p0→1         |            |           
W B0 wait_p0 (cur_p0)
0003 |                    | I0:b0, OB0 |           
0004 |                    | I1:b1, OB0 |           
0005 |                    | I2:b2, OB0 |           
-----------------------------------------------------------------
0006 |                    | I3:b3, OB0 | b0→C0     
0007 |                    | I4:b0, OB0 | b1→C1     
0008 |                    | I5:b1, OB0 | b2→C2     
0009 |                    | I6:b2, OB0 | b3→C3     
0010 | O3:B0 p0→1         | I7:b3, OB0 | b0→C4     
0011 |                    | I8:b0, OB0 | b1→C5     
0012 |                    | I9:b1, OB0 | b2→C6     
-----------------------------------------------------------------
0013 |                    | I3:b2, OB1 | b3→C0     
0014 |                    | I4:b3, OB1 | b0→C1     
0015 |                    | I5:b0, OB1 | b1→C2     
0016 |        